# Demo 6 — La scala temperata

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-3485c94980f2f6df")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        url = str(location.origin) + prefisso + "/_jupyterlite/files/audiolab-course.zip?v=3485c94980f2f6df"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
y, sr = al.load_audio(AUDIO / "C-major-scale.wav", sr=SR)
nomi = ["do", "do#", "re", "re#", "mi", "fa", "fa#", "sol", "sol#", "la", "la#", "si"]
freq_teoriche = al.midi_to_frequency(60) * 2 ** (np.arange(12) / 12)

print(f"{'semitono':9s} {'teorica [Hz]':>13s} {'fattore':>9s}")
for i, nome in enumerate(nomi):
    fattore = freq_teoriche[i] / freq_teoriche[i - 1] if i else float("nan")
    print(f"{nome:9s} {freq_teoriche[i]:13.2f} {fattore:9.4f}")
print("\nfattore atteso per un semitono: 2^(1/12) =", round(2 ** (1 / 12), 4))
print("dopo 12 semitoni:", round(freq_teoriche[11] / freq_teoriche[0] * 2 ** (1 / 12), 3), "(cioè 2 volte do4)")

display(al.safe_audio(y, sr, normalize=False))

# Le frequenze teoriche della scala maggiore: toni e semitoni, non 12 note cromatiche.
passi = np.array([0, 2, 4, 5, 7, 9, 11, 12])
nomi_scala = ["do4", "re4", "mi4", "fa4", "sol4", "la4", "si4", "do5"]
frequenze_scala = al.midi_to_frequency(60) * 2 ** (passi / 12)
fig = go.Figure(go.Scatter(x=nomi_scala, y=frequenze_scala, mode="lines+markers",
                           line=dict(color="#1769C2"), marker=dict(size=8)))
fig.update_yaxes(title_text="frequenza teorica [Hz]")
fig.update_xaxes(title_text="nota")
fig.update_layout(height=340, title="Scala di do maggiore: frequenze teoriche")
fig.show()
# La registrazione completa verrà analizzata nel tempo nella Demo 10.
